# Bloc 2 — Comprendre pytest, monkeypatch et mock

pytest se lance **en ligne de commande** sur des **fichiers** `test_*.py`. Dans ce notebook :
- les cellules `%%writefile` **créent** les fichiers dans le dossier `tests_demo/` ;
- la fonction `pytest(...)` **lance** pytest sur ces fichiers et affiche le résultat.

Tu peux donc modifier un test, réexécuter la cellule `%%writefile`, puis relancer pytest.

| # | Partie | Idée clé |
|---|---|---|
| 1 | Le code à tester | un petit module `boutique.py` |
| 2 | Bases de pytest | `assert`, `approx`, `raises`, lire un échec |
| 3 | `parametrize` | un test, plusieurs jeux de données |
| 4 | Fixtures | préparer / nettoyer, `tmp_path`, `conftest.py` |
| 5 | `monkeypatch` | **remplacer** temporairement une variable d'env, une fonction, un attribut |
| 6 | `mock` | **remplacer et espionner** : « a-t-on appelé ça, avec quoi ? » |
| 7 | Application au bloc 2 | tester `etl.py` et `ingestion.py` pour de vrai |
| 8 | Antisèche | |

In [1]:
import subprocess
import sys
from pathlib import Path

Path("tests_demo").mkdir(exist_ok=True)

def pytest(*args):
    """Lance `python -m pytest <args>` dans tests_demo/ et affiche la sortie."""
    result = subprocess.run(
        [sys.executable, "-m", "pytest", *args, "-p", "no:cacheprovider"],
        cwd="tests_demo", capture_output=True, text=True, encoding="utf-8",
    )
    print(result.stdout[-6000:], result.stderr[-2000:])

## 1. Le code à tester

Un module volontairement simple, mais avec les cas qui posent problème en test :
- une fonction **pure** (`prix_ttc`) : facile à tester ;
- une fonction qui lit une **variable d'environnement** ;
- une fonction qui lit un **fichier** ;
- une fonction qui appelle une **API sur Internet** (lente, payante, instable, absente en CI…) ;
- une fonction qui **envoie une alerte** (on ne veut pas envoyer de vrais e-mails pendant les tests).

In [2]:
%%writefile tests_demo/boutique.py
"""Petit module métier servant d'exemple pour les tests."""
import json
import os
from urllib.request import urlopen

TVA = 0.20


def prix_ttc(prix_ht: float, taux: float = TVA) -> float:
    if prix_ht < 0:
        raise ValueError("Le prix HT doit être positif")
    return round(number=prix_ht * (1 + taux), ndigits=2)


def get_database_url() -> str:
    url = os.getenv(key="DATABASE_URL")
    if url is None:
        raise RuntimeError("DATABASE_URL non définie")
    return url


def charger_commandes(chemin) -> list[dict]:
    with open(file=chemin, encoding="utf-8") as f:
        return json.load(fp=f)


def get_taux_change(devise: str) -> float:
    """Appelle une API externe : réseau, lenteur, quota... à ne PAS appeler en test."""
    with urlopen(url=f"https://api.exemple.com/taux/{devise}", timeout=5) as reponse:
        return json.load(fp=reponse)["taux"]


def convertir(montant_eur: float, devise: str) -> float:
    return round(number=montant_eur * get_taux_change(devise=devise), ndigits=2)


class Notificateur:
    """Envoie de vrais messages (e-mail, Slack...)."""
    def envoyer(self, destinataire: str, message: str) -> bool:
        raise NotImplementedError("envoi réel")


def traiter_commande(commande: dict, notificateur: Notificateur) -> float:
    """Calcule le total et alerte l'équipe fraude au-delà de 1000 €."""
    total = sum(l["prix"] * l["qte"] for l in commande["lignes"])
    if total > 1000:
        notificateur.envoyer(destinataire="fraude@shop.fr", message=f"Commande {commande['id']} : {total} €")
    return total

Overwriting tests_demo/boutique.py


## 2. Les bases de pytest

**Conventions** (c'est ainsi que pytest *découvre* les tests, sans rien déclarer) :
- fichiers `test_*.py` (ou `*_test.py`) ;
- fonctions `test_*` (ou méthodes `test_*` dans une classe `Test*`) ;
- un simple `assert` : si l'expression est fausse, le test échoue. Pas besoin de
  `self.assertEqual` comme avec `unittest`.

Structure recommandée d'un test : **AAA — Arrange / Act / Assert**.

In [3]:
%%writefile tests_demo/test_01_bases.py
import pytest

from boutique import prix_ttc


def test_prix_ttc_taux_par_defaut():
    # Arrange : préparer les données
    prix_ht = 100
    # Act : appeler le code testé
    resultat = prix_ttc(prix_ht=prix_ht)
    # Assert : vérifier
    assert resultat == 120


def test_prix_ttc_taux_reduit():
    assert prix_ttc(prix_ht=100, taux=0.055) == 105.5


def test_flottants_avec_approx():
    # 0.1 + 0.2 == 0.30000000000000004 : on compare avec une tolérance
    assert 0.1 + 0.2 == pytest.approx(expected=0.3)


def test_prix_negatif_leve_une_erreur():
    # On vérifie qu'une exception EST levée (et son message, via une regex)
    with pytest.raises(expected_exception=ValueError, match="positif"):
        prix_ttc(prix_ht=-5)

Overwriting tests_demo/test_01_bases.py


In [4]:
pytest("test_01_bases.py", "-v")

============================= test session starts ==============================
platform linux -- Python 3.12.5, pytest-9.1.1, pluggy-1.7.0 -- /mnt/c/Users/Alexandre/Desktop/fiches_revision/.venv/bin/python
rootdir: /mnt/c/Users/Alexandre/Desktop/fiches_revision/bloc2/tests_demo
plugins: platformdirs-4.13.5
collecting ... collected 4 items

test_01_bases.py::test_prix_ttc_taux_par_defaut PASSED                   [ 25%]
test_01_bases.py::test_prix_ttc_taux_reduit PASSED                       [ 50%]
test_01_bases.py::test_flottants_avec_approx PASSED                      [ 75%]
test_01_bases.py::test_prix_negatif_leve_une_erreur PASSED               [100%]

============================== 4 passed in 0.05s ===============================
 


`-v` (verbose) affiche chaque test. Autres options utiles :
`-q` (silencieux), `-k "taux"` (ne lance que les tests dont le nom contient *taux*),
`-x` (s'arrête au premier échec), `-s` (affiche les `print`), `--lf` (relance les derniers échecs).

### À quoi ressemble un échec ?
pytest réécrit les `assert` pour montrer **les valeurs réelles** :

In [5]:
%%writefile tests_demo/test_00_echec.py
from boutique import prix_ttc


def test_qui_echoue():
    commande = {"id": 1, "total": prix_ttc(prix_ht=100)}
    assert commande == {"id": 1, "total": 121}

Writing tests_demo/test_00_echec.py


In [6]:
pytest("test_00_echec.py", "-q")

F                                                                        [100%]
=================================== FAILURES ===================================
_______________________________ test_qui_echoue ________________________________

    def test_qui_echoue():
        commande = {"id": 1, "total": prix_ttc(100)}
>       assert commande == {"id": 1, "total": 121}
E       AssertionError: assert {'id': 1, 'total': 120.0} == {'id': 1, 'total': 121}
E         
E         Omitting 1 identical items, use -vv to show
E         Differing items:
E         {'total': 120.0} != {'total': 121}
E         Use -v to get more diff

test_00_echec.py:6: AssertionError
=========================== short test summary info ============================
FAILED test_00_echec.py::test_qui_echoue - AssertionError: assert {'id': 1, 'total': 120.0} == {'id': 1, 'total': 121}
1 failed in 0.40s
 


On lit : la ligne fautive, puis le *diff* `120.0 != 121`. On supprime ce fichier pour la suite.

In [7]:
Path("tests_demo/test_00_echec.py").unlink()

## 3. `parametrize` : un test, plusieurs cas

Au lieu de copier-coller le même test, on le paramètre. Chaque jeu de données devient
**un test distinct** dans le rapport (et un échec n'empêche pas les autres de s'exécuter).

In [9]:
%%writefile tests_demo/test_02_parametrize.py
import pytest

from boutique import prix_ttc


@pytest.mark.parametrize(
    argnames="prix_ht, taux, attendu",
    argvalues=[
        (100, 0.20, 120.0),
        (100, 0.055, 105.5),
        (0, 0.20, 0.0),
        (19.99, 0.20, 23.99),
    ],
    ids=["normal", "reduit", "gratuit", "arrondi"],
)
def test_prix_ttc(prix_ht, taux, attendu):
    assert prix_ttc(prix_ht=prix_ht, taux=taux) == attendu


@pytest.mark.parametrize(argnames="prix_invalide", argvalues=[-1, -0.01, -1000])
def test_prix_invalides(prix_invalide):
    with pytest.raises(expected_exception=ValueError):
        prix_ttc(prix_ht=prix_invalide)

Overwriting tests_demo/test_02_parametrize.py


In [10]:
pytest("test_02_parametrize.py", "-v")

============================= test session starts ==============================
platform linux -- Python 3.12.5, pytest-9.1.1, pluggy-1.7.0 -- /mnt/c/Users/Alexandre/Desktop/fiches_revision/.venv/bin/python
rootdir: /mnt/c/Users/Alexandre/Desktop/fiches_revision/bloc2/tests_demo
plugins: platformdirs-4.13.5
collecting ... collected 7 items

test_02_parametrize.py::test_prix_ttc[normal] PASSED                     [ 14%]
test_02_parametrize.py::test_prix_ttc[reduit] PASSED                     [ 28%]
test_02_parametrize.py::test_prix_ttc[gratuit] PASSED                    [ 42%]
test_02_parametrize.py::test_prix_ttc[arrondi] PASSED                    [ 57%]
test_02_parametrize.py::test_prix_invalides[-1] PASSED                   [ 71%]
test_02_parametrize.py::test_prix_invalides[-0.01] PASSED                [ 85%]
test_02_parametrize.py::test_prix_invalides[-1000] PASSED                [100%]

============================== 7 passed in 0.04s ===============================
 


## 4. Les fixtures

Une **fixture** est une fonction décorée par `@pytest.fixture` qui **prépare** quelque chose
(données, fichier, connexion BDD…). Un test la reçoit **en la nommant dans ses paramètres** :
pytest l'exécute et injecte le résultat (*injection de dépendances*).

- Avec `yield` : le code **après** le `yield` est exécuté à la fin du test (*teardown*),
  même si le test échoue → idéal pour fermer une connexion, supprimer un fichier.
- `scope="function"` (défaut, recréée à chaque test), `"module"`, `"session"` (une seule fois).
- Fixtures **intégrées** utiles : `tmp_path` (dossier temporaire unique), `monkeypatch`,
  `capsys` (capturer `print`), `caplog` (capturer les logs).
- Placées dans un fichier **`conftest.py`**, elles sont disponibles pour tous les tests
  du dossier **sans import**.

In [11]:
%%writefile tests_demo/conftest.py
import sys
from pathlib import Path

import pytest

# Rend importables les scripts du bloc 2 (etl.py, ingestion.py...) pour la partie 7
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))


@pytest.fixture
def commande_simple():
    """Disponible dans TOUS les fichiers de test du dossier, sans import."""
    return {"id": 42, "lignes": [{"prix": 10.0, "qte": 2}, {"prix": 5.0, "qte": 1}]}


@pytest.fixture
def grosse_commande():
    return {"id": 99, "lignes": [{"prix": 600.0, "qte": 2}]}

Overwriting tests_demo/conftest.py


In [12]:
%%writefile tests_demo/test_03_fixtures.py
import json

import pytest

from boutique import charger_commandes


@pytest.fixture
def fichier_commandes(tmp_path):
    """Une fixture peut utiliser une autre fixture (ici tmp_path)."""
    chemin = tmp_path / "commandes.json"
    chemin.write_text(data=json.dumps(obj=[{"id": 1}, {"id": 2}]), encoding="utf-8")
    print(f"\n[setup] fichier créé : {chemin.name}")
    yield chemin                      # <- valeur injectée dans le test
    print("[teardown] nettoyage après le test")


def test_charger_commandes(fichier_commandes):
    commandes = charger_commandes(chemin=fichier_commandes)
    assert [c["id"] for c in commandes] == [1, 2]


def test_fichier_absent(tmp_path):
    with pytest.raises(expected_exception=FileNotFoundError):
        charger_commandes(chemin=tmp_path / "inexistant.json")


def test_fixture_du_conftest(commande_simple):
    # commande_simple vient de conftest.py, aucun import nécessaire
    assert len(commande_simple["lignes"]) == 2

Overwriting tests_demo/test_03_fixtures.py


In [13]:
pytest("test_03_fixtures.py", "-v", "-s")

============================= test session starts ==============================
platform linux -- Python 3.12.5, pytest-9.1.1, pluggy-1.7.0 -- /mnt/c/Users/Alexandre/Desktop/fiches_revision/.venv/bin/python
rootdir: /mnt/c/Users/Alexandre/Desktop/fiches_revision/bloc2/tests_demo
plugins: platformdirs-4.13.5
collecting ... collected 3 items

test_03_fixtures.py::test_charger_commandes 
[setup] fichier créé : commandes.json
PASSED[teardown] nettoyage après le test

test_03_fixtures.py::test_fichier_absent PASSED
test_03_fixtures.py::test_fixture_du_conftest PASSED

============================== 3 passed in 0.05s ===============================
 


Avec `-s`, on voit l'ordre : *setup* → test → *teardown*.

## 5. `monkeypatch` : remplacer temporairement

**Problème :** le code dépend de choses qu'on ne contrôle pas en test (variables
d'environnement, API externe, date du jour, chemin…).

**Solution :** la fixture intégrée `monkeypatch` **remplace** une valeur pendant la durée
**d'un seul test**, puis **restaure automatiquement** l'original. Aucun effet de bord sur
les autres tests.

| Méthode | Usage |
|---|---|
| `monkeypatch.setenv("VAR", "val")` / `delenv("VAR")` | variables d'environnement |
| `monkeypatch.setattr(objet, "nom", valeur)` | remplacer un attribut / une fonction |
| `monkeypatch.setattr("module.nom", valeur)` | idem, avec une chaîne |
| `monkeypatch.setitem(dico, cle, valeur)` | modifier un dictionnaire |
| `monkeypatch.chdir(chemin)` | changer de dossier courant |

On remplace par **ce qu'on veut** (souvent une petite fonction `lambda` ou une fonction écrite
à la main, qu'on appelle un *stub* ou un *fake*).

In [15]:
%%writefile tests_demo/test_04_monkeypatch.py
import io
import json

import pytest

import boutique


# --- Variables d'environnement ---------------------------------------------
def test_database_url_definie(monkeypatch):
    monkeypatch.setenv(name="DATABASE_URL", value="sqlite:///test.db")
    assert boutique.get_database_url() == "sqlite:///test.db"


def test_database_url_absente(monkeypatch):
    # raising=False : pas d'erreur si la variable n'existait pas déjà
    monkeypatch.delenv(name="DATABASE_URL", raising=False)
    with pytest.raises(expected_exception=RuntimeError, match="non définie"):
        boutique.get_database_url()


# --- Remplacer une fonction (éviter l'appel réseau) ------------------------
def test_convertir_sans_reseau(monkeypatch):
    # convertir() appelle get_taux_change() -> on la remplace par un stub
    monkeypatch.setattr(target=boutique, name="get_taux_change", value=lambda devise: 1.10)
    assert boutique.convertir(montant_eur=100, devise="USD") == 110.0


def test_stub_qui_verifie_ses_arguments(monkeypatch):
    appels = []                                  # on note les appels à la main

    def faux_taux(devise):
        appels.append(devise)
        return {"USD": 1.10, "GBP": 0.85}[devise]

    monkeypatch.setattr(target=boutique, name="get_taux_change", value=faux_taux)
    assert boutique.convertir(montant_eur=100, devise="GBP") == 85.0
    assert appels == ["GBP"]
    # (c'est fastidieux : unittest.mock fait ce travail d'espion pour nous, cf. partie 6)


# --- Remplacer une dépendance plus bas niveau (urlopen) --------------------
def test_get_taux_change_avec_fausse_reponse_http(monkeypatch):
    def faux_urlopen(url, timeout):
        assert url.endswith("/USD")
        return io.BytesIO(initial_bytes=json.dumps(obj={"taux": 1.08}).encode())  # objet "fichier" avec with

    # boutique a fait "from urllib.request import urlopen" :
    # le nom à remplacer est donc boutique.urlopen (voir la règle « où patcher » en partie 6)
    monkeypatch.setattr("boutique.urlopen", faux_urlopen)
    assert boutique.get_taux_change(devise="USD") == 1.08


# --- Piège : les valeurs par défaut sont figées à la définition ------------
def test_piege_valeur_par_defaut(monkeypatch):
    monkeypatch.setattr(target=boutique, name="TVA", value=0.055)
    # prix_ttc(prix_ht, taux=TVA) : la valeur par défaut a été calculée À L'IMPORT (0.20)
    assert boutique.prix_ttc(prix_ht=100) == 120.0       # et non 105.5 !


def test_restauration_automatique():
    # Le test précédent a modifié boutique.TVA : monkeypatch l'a remis en place
    assert boutique.TVA == 0.20

Writing tests_demo/test_04_monkeypatch.py


In [14]:
pytest("test_04_monkeypatch.py", "-v")

============================= test session starts =============================
platform win32 -- Python 3.10.11, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\Alexandre\Desktop\fiches_revision\bloc2\.venv\Scripts\python.exe
rootdir: C:\Users\Alexandre\Desktop\fiches_revision\bloc2\tests_demo
plugins: platformdirs-4.12.4
collecting ... collected 7 items

test_04_monkeypatch.py::test_database_url_definie PASSED                 [ 14%]
test_04_monkeypatch.py::test_database_url_absente PASSED                 [ 28%]
test_04_monkeypatch.py::test_convertir_sans_reseau PASSED                [ 42%]
test_04_monkeypatch.py::test_stub_qui_verifie_ses_arguments PASSED       [ 57%]
test_04_monkeypatch.py::test_get_taux_change_avec_fausse_reponse_http PASSED [ 71%]
test_04_monkeypatch.py::test_piege_valeur_par_defaut PASSED              [ 85%]
test_04_monkeypatch.py::test_restauration_automatique PASSED             [100%]

============================== 7 passed in 0.02s ==============================
 


**À retenir sur le piège des valeurs par défaut :** `def f(x=CONSTANTE)` évalue `CONSTANTE`
**une seule fois**, quand Python lit le `def`. Patcher la constante ensuite ne change rien.
On retrouve exactement ce cas dans `create_db.py` du bloc 2 (`url: str = DATABASE_URL`) :
la partie 7 montre comment le contourner.

## 6. `unittest.mock` : remplacer **et espionner**

Un `Mock` est un objet « caméléon » de la bibliothèque standard :
- on peut appeler **n'importe quelle méthode** dessus, il ne plante pas ;
- on choisit ce qu'il **renvoie** (`return_value`) ou ce qu'il **fait** (`side_effect`) ;
- il **enregistre tous ses appels** → on peut vérifier ensuite *si* et *comment* il a été appelé.

| Outil | Rôle |
|---|---|
| `Mock()` / `MagicMock()` | faux objet (`MagicMock` gère aussi `with`, `len`, `[]`, itération…) |
| `.return_value = x` | valeur renvoyée à l'appel |
| `.side_effect = Exception(...)` | lever une erreur à l'appel |
| `.side_effect = [a, b, c]` | renvoyer a, puis b, puis c |
| `.side_effect = fonction` | calculer la réponse selon les arguments |
| `assert_called_once_with(...)` | appelé exactement une fois, avec ces arguments |
| `assert_not_called()`, `.call_count`, `.call_args` | autres vérifications |
| `patch("module.nom")` | remplace `module.nom` par un `MagicMock` le temps d'un test (décorateur ou `with`) |
| `Mock(spec=Classe)` / `create_autospec` | le faux respecte l'interface de la vraie classe |

### 6.1 Le Mock tout seul

In [28]:
from unittest.mock import Mock

api = Mock()
api.get_taux.return_value = 1.10

print(api.get_taux("USD"))            # 1.1
print(api.get_taux.call_count)        # 1
print(api.get_taux.call_args)         # call('USD')
api.get_taux.assert_called_once_with("USD")   # OK, ne lève rien

print(api.methode_inventee(1, 2))     # ne plante pas : renvoie un autre Mock

1.1
1
call('USD')
<Mock name='mock.methode_inventee()' id='140650519244880'>


In [29]:
api.get_taux.side_effect = [1.10, 1.12, TimeoutError("API indisponible")]
print(api.get_taux("USD"), api.get_taux("USD"))
try:
    api.get_taux("USD")
except TimeoutError as e:
    print("Erreur simulée :", e)

1.1 1.12
Erreur simulée : API indisponible


### 6.2 Dans des tests

Deux usages typiques :
1. **Injecter** un mock quand le code reçoit sa dépendance en paramètre (`traiter_commande(commande, notificateur)`) :
   c'est le cas le plus simple et le plus propre.
2. **Patcher** avec `patch(...)` quand la dépendance est appelée « en dur » dans le code.

In [31]:
%%writefile tests_demo/test_05_mock.py
import urllib.request
from unittest.mock import MagicMock, Mock, create_autospec, patch

import pytest

import boutique
from boutique import Notificateur, traiter_commande


# --- 1. Injection : le notificateur est passé en paramètre -----------------
def test_pas_d_alerte_pour_petite_commande(commande_simple):
    notif = Mock()
    total = traiter_commande(commande=commande_simple, notificateur=notif)
    assert total == 25.0
    notif.envoyer.assert_not_called()


def test_alerte_fraude_pour_grosse_commande(grosse_commande):
    notif = Mock()
    traiter_commande(commande=grosse_commande, notificateur=notif)
    # On vérifie le COMPORTEMENT : qui a été prévenu, avec quel message
    notif.envoyer.assert_called_once_with(destinataire="fraude@shop.fr", message="Commande 99 : 1200.0 €")


def test_spec_detecte_les_fautes_de_frappe():
    notif = create_autospec(spec=Notificateur, instance=True)
    with pytest.raises(expected_exception=AttributeError):
        notif.envoie("x", "y")          # faute : la vraie méthode s'appelle envoyer
    with pytest.raises(expected_exception=TypeError):
        notif.envoyer("seulement un argument")   # mauvaise signature détectée


# --- 2. patch en décorateur : le mock arrive en paramètre -------------------
@patch(target="boutique.get_taux_change", return_value=1.10)
def test_convertir_avec_patch(mock_taux):
    assert boutique.convertir(montant_eur=100, devise="USD") == 110.0
    mock_taux.assert_called_once_with(devise="USD")


# --- patch en context manager + side_effect = exception ---------------------
def test_convertir_propage_l_erreur_api():
    with patch(target="boutique.get_taux_change", side_effect=TimeoutError("API HS")):
        with pytest.raises(expected_exception=TimeoutError):
            boutique.convertir(montant_eur=100, devise="USD")


# --- MagicMock pour simuler un objet utilisé avec "with" --------------------
def test_get_taux_change_mock_http():
    fausse_reponse = MagicMock()
    fausse_reponse.__enter__.return_value.read.return_value = b'{"taux": 0.85}'
    with patch(target="boutique.urlopen", return_value=fausse_reponse) as mock_urlopen:
        assert boutique.get_taux_change(devise="GBP") == 0.85
    mock_urlopen.assert_called_once_with(url="https://api.exemple.com/taux/GBP", timeout=5)


# --- LA règle : patcher là où le nom est UTILISÉ, pas là où il est défini --
def test_ou_patcher():
    with patch(target="urllib.request.urlopen") as mauvais:
        # boutique a copié la référence à l'import (from urllib.request import urlopen) :
        # patcher urllib.request ne touche PAS boutique.urlopen -> le vrai réseau serait appelé
        assert boutique.urlopen is not mauvais
    with patch(target="boutique.urlopen") as bon:
        assert boutique.urlopen is bon

Overwriting tests_demo/test_05_mock.py


In [32]:
pytest("test_05_mock.py", "-v")

============================= test session starts ==============================
platform linux -- Python 3.12.5, pytest-9.1.1, pluggy-1.7.0 -- /mnt/c/Users/Alexandre/Desktop/fiches_revision/.venv/bin/python
rootdir: /mnt/c/Users/Alexandre/Desktop/fiches_revision/bloc2/tests_demo
plugins: platformdirs-4.13.5
collecting ... collected 7 items

test_05_mock.py::test_pas_d_alerte_pour_petite_commande PASSED           [ 14%]
test_05_mock.py::test_alerte_fraude_pour_grosse_commande PASSED          [ 28%]
test_05_mock.py::test_spec_detecte_les_fautes_de_frappe PASSED           [ 42%]
test_05_mock.py::test_convertir_avec_patch PASSED                        [ 57%]
test_05_mock.py::test_convertir_propage_l_erreur_api PASSED              [ 71%]
test_05_mock.py::test_get_taux_change_mock_http PASSED                   [ 85%]
test_05_mock.py::test_ou_patcher PASSED                                  [100%]

============================== 7 passed in 0.08s ===============================
 


### 6.3 La règle « où patcher » (la plus grande source d'erreurs)

```python
# boutique.py
from urllib.request import urlopen   # crée un NOUVEAU nom : boutique.urlopen
```
`patch("urllib.request.urlopen")` remplace le nom dans `urllib.request`… mais `boutique` garde sa
propre référence vers la vraie fonction. **On patche le nom dans le module qui l'utilise :**
`patch("boutique.urlopen")`.

(Si `boutique.py` avait fait `import urllib.request` puis `urllib.request.urlopen(...)`,
il aurait fallu patcher `"urllib.request.urlopen"`.)

### 6.4 monkeypatch ou mock ?

| | `monkeypatch` (pytest) | `unittest.mock` |
|---|---|---|
| Rôle | **remplacer** puis restaurer | remplacer **+ enregistrer les appels** |
| Remplaçant | ce que tu veux (valeur, lambda, fonction) | un objet `Mock` configurable |
| Vérifier les appels | à la main (liste d'appels) | `assert_called_once_with`, `call_args`… |
| Variables d'env | `setenv` / `delenv` (très simple) | `patch.dict(os.environ, {...})` |
| Restauration | automatique à la fin du test | automatique (fin du `with` / du test décoré) |

En pratique on **combine** : `monkeypatch.setattr(module, "fonction", Mock(return_value=...))`
remplace avec monkeypatch et espionne avec un Mock.

Le plugin `pytest-mock` fournit aussi une fixture `mocker` (`mocker.patch(...)`), même API que
`patch` mais sans décorateur ni `with`.

**Attention à l'excès de mocks :** un test qui mocke tout ne teste plus rien. On mocke
les **frontières** (réseau, e-mail, horloge, hasard, services payants), pas sa propre logique.

## 7. Application au bloc 2

Tests réels de `etl.py` et `ingestion.py`, avec des données **fabriquées** (rapides et
maîtrisées) au lieu des 179 Mo de `orders.json`.

Point intéressant : `ingest()` appelle `create_database()` **sans argument**, et
`create_database(url=DATABASE_URL)` a figé l'URL à l'import (le piège de la partie 5).
`monkeypatch.setenv("DATABASE_URL", ...)` ne suffirait donc pas : on remplace
`ingestion.create_database` (là où il est **utilisé**) pour pointer vers une base SQLite
temporaire. La vraie base `data/shop.db` n'est jamais touchée.

In [19]:
%%writefile tests_demo/test_06_bloc2.py
import json

import pytest
from sqlalchemy import func, select
from sqlalchemy.orm import Session

import etl
import ingestion
from create_db import Order, OrderItem, Product, create_database


def ligne(order_id=1, produit="Smart watch ", qte=1, prix=100.0, statut="COMPLETE", **extra):
    """Fabrique une ligne au format du JSON brut."""
    rec = {
        "sales": qte * prix, "type": "TRANSFER", "market": "Europe",
        "date order": "2017-01-10T10:00:00", "date shipping": "2017-01-12T10:00:00",
        "id": order_id, "status": statut, "quantity": qte,
        "loc": {"country": "France", "state": "IDF", "region": "Western Europe", "city": "Paris"},
        "product": {"name": produit, "category_id": 1, "category_name": "Electronics", "price": prix},
        "customer": {"id": 7, "firstname": "Ada", "lastname": "Lovelace",
                     "loc": {"country": "EE. UU.", "state": "PR", "city": "Caguas",
                             "zipcode": 725.0, "street": "1 rue X"}},
    }
    rec.update(extra)
    return rec


@pytest.fixture
def records():
    return [
        ligne(order_id=1, produit="Smart watch "),
        ligne(order_id=1, produit="Smart watch "),               # doublon strict
        ligne(order_id=1, produit="Tente", qte=2, prix=50.0),
        ligne(order_id=2, statut="SUSPECTED_FRAUD"),
    ]


# --- ETL -------------------------------------------------------------------
def test_transform_nettoie(records):
    df = etl.transform(records=records)
    assert len(df) == 3                                   # doublon supprimé
    assert set(df["product_name"]) == {"Smart watch", "Tente"}   # espaces retirés
    assert df["customer_zipcode"].iloc[0] == "00725"      # float -> texte 5 chiffres
    assert df.loc[df["order_id"] == 2, "is_fraud"].item() == 1
    assert (df["shipping_days"] == 2).all()


def test_validate_rejette_montant_incoherent(records):
    records.append(ligne(order_id=3, sales=999.0))        # sales != qte x prix
    df = etl.transform(records=records)
    assert 3 not in df["order_id"].values


def test_lastname_manquant_impute(records):
    del records[0]["customer"]["lastname"]
    df = etl.transform(records=records[:1])
    assert df["customer_lastname"].item() == "Inconnu"


def test_extract_fichier_absent(tmp_path):
    with pytest.raises(expected_exception=FileNotFoundError):
        etl.extract(path=tmp_path / "absent.json")


def test_extract_refuse_un_objet(tmp_path):
    chemin = tmp_path / "objet.json"
    chemin.write_text(data=json.dumps(obj={"pas": "une liste"}), encoding="utf-8")
    with pytest.raises(expected_exception=ValueError, match="liste"):
        etl.extract(path=chemin)


# --- Ingestion dans une base temporaire --------------------------------------
@pytest.fixture
def base_temporaire(tmp_path, monkeypatch):
    url = f"sqlite:///{tmp_path / 'test.db'}"
    monkeypatch.setattr(target=ingestion, name="create_database", value=lambda: create_database(url=url))
    return create_database(url=url)       # engine pour interroger la base dans le test


def compter(engine, modele):
    with Session(bind=engine) as s:
        return s.scalar(statement=select(func.count()).select_from(modele))


def test_ingestion(records, base_temporaire):
    ingestion.ingest(df=etl.transform(records=records))
    assert compter(engine=base_temporaire, modele=Order) == 2
    assert compter(engine=base_temporaire, modele=OrderItem) == 3
    assert compter(engine=base_temporaire, modele=Product) == 2


def test_ingestion_idempotente(records, base_temporaire):
    df = etl.transform(records=records)
    ingestion.ingest(df=df)
    ingestion.ingest(df=df)                  # relancer ne doit pas créer de doublons
    assert compter(engine=base_temporaire, modele=OrderItem) == 3

Overwriting tests_demo/test_06_bloc2.py


In [20]:
pytest("test_06_bloc2.py", "-v")

============================= test session starts =============================
platform win32 -- Python 3.10.11, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\Alexandre\Desktop\fiches_revision\bloc2\.venv\Scripts\python.exe
rootdir: C:\Users\Alexandre\Desktop\fiches_revision\bloc2\tests_demo
plugins: platformdirs-4.12.4
collecting ... collected 7 items

test_06_bloc2.py::test_transform_nettoie PASSED                          [ 14%]
test_06_bloc2.py::test_validate_rejette_montant_incoherent PASSED        [ 28%]
test_06_bloc2.py::test_lastname_manquant_impute PASSED                   [ 42%]
test_06_bloc2.py::test_extract_fichier_absent PASSED                     [ 57%]
test_06_bloc2.py::test_extract_refuse_un_objet PASSED                    [ 71%]
test_06_bloc2.py::test_ingestion PASSED                                  [ 85%]
test_06_bloc2.py::test_ingestion_idempotente PASSED                      [100%]

============================== 7 passed in 0.67s ==============================
 


**Anecdote utile pour l'oral :** à sa première exécution, `test_lastname_manquant_impute`
a **échoué** et révélé un vrai bug de `etl.py`. Si *aucune* ligne d'un lot n'a de
`lastname`, `json_normalize` ne crée pas la colonne et l'ETL plantait au lieu d'imputer
`"Inconnu"`. Sur le gros fichier, ce cas n'arrive jamais (8 manquants sur 180 000), mais il
arriverait avec un chargement incrémental de petits lots. Correction : `OPTIONAL_COLUMNS`
dans `flatten()`. C'est exactement le rôle des tests : couvrir les **cas limites**.

### Tout lancer d'un coup

In [21]:
pytest("-q")

...................................                                      [100%]
35 passed in 0.71s
 


En dehors du notebook, la commande équivalente (depuis `bloc2/`) est :
```bash
.venv/Scripts/python -m pytest tests_demo -v
```

## 8. Antisèche

```python
import pytest
from unittest.mock import Mock, MagicMock, patch, create_autospec

# Bases
assert f(2) == 4
assert x == pytest.approx(0.3)
with pytest.raises(ValueError, match="positif"): f(-1)

# Plusieurs cas
@pytest.mark.parametrize("entree, attendu", [(1, 2), (2, 4)])
def test_f(entree, attendu): assert f(entree) == attendu

# Fixture (setup / teardown)
@pytest.fixture
def connexion():
    c = ouvrir(); yield c; c.close()

# monkeypatch : remplacer temporairement
def test_x(monkeypatch, tmp_path):
    monkeypatch.setenv("DATABASE_URL", "sqlite://")
    monkeypatch.setattr(module, "fonction", lambda *a: 42)

# mock : remplacer + espionner
m = Mock(return_value=42)            # ou side_effect=Exception / [1, 2] / fonction
m.assert_called_once_with("arg")
@patch("module_qui_utilise.nom")      # patcher LÀ OÙ C'EST UTILISÉ
def test_y(mock_nom): ...
```

**Questions classiques à l'oral**
- *Test unitaire vs test d'intégration ?* Unitaire : une fonction isolée, dépendances
  remplacées (rapide). Intégration : plusieurs briques ensemble (ex. ETL + vraie base SQLite
  temporaire, comme en partie 7).
- *Pourquoi mocker ?* Rendre les tests **rapides, reproductibles, hors ligne** et sans effet
  de bord (pas de vrai e-mail, pas d'appel d'API payante).
- *Stub vs mock ?* Un stub **renvoie** une valeur fixe ; un mock permet en plus de **vérifier
  les appels** reçus.
- *Pourquoi `tmp_path` ?* Chaque test a son propre dossier : pas de fichiers qui traînent,
  pas d'interférence entre tests.
- *Couverture ?* `pip install pytest-cov` puis `pytest --cov=. --cov-report=term-missing`.